In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Tải dữ liệu xe cũ Car Dekho
df = pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")

print("Kích thước dữ liệu gốc:", df.shape)
print("Thông tin cột:")
print(df.info())
df.head()

Kích thước dữ liệu gốc: (4340, 8)
Thông tin cột:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB
None


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [2]:
# 2. Xử lý trùng lặp và thiếu
df_clean = df.drop_duplicates().dropna().reset_index(drop=True)
print(f"Số dòng sau khi làm sạch: {len(df_clean)}")

# 3. Định nghĩa nhóm thuộc tính
num_cols = ["selling_price", "km_driven", "year"]
cat_cols = ["fuel", "seller_type", "transmission", "owner"]

# 4. Chuẩn hóa Min-Max Scaling cho thuộc tính số về [0, 1]
scaler = MinMaxScaler()
df_scaled_num = pd.DataFrame(
    scaler.fit_transform(df_clean[num_cols]),
    columns=[f"{col}_scaled" for col in num_cols],
)

# Ghép nối lại thành cơ sở trường hợp (Case Base)
case_base = pd.concat([df_clean, df_scaled_num], axis=1)
print("Cơ sở trường hợp đã hoàn tất chuẩn hóa.")
case_base.head(2)

Số dòng sau khi làm sạch: 3577
Cơ sở trường hợp đã hoàn tất chuẩn hóa.


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,selling_price_scaled,km_driven_scaled,year_scaled
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,0.004505,0.086783,0.535714
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,0.012950,0.061988,0.535714


In [3]:
# 5. Hàm tính độ tương đồng đa thuộc tính
def calculate_similarity(query_item, case_df, weights=None):
    if weights is None:
        # Mặc định chia đều trọng số cho các thuộc tính
        weights = {
            "selling_price": 0.3,
            "km_driven": 0.2,
            "year": 0.2,
            "fuel": 0.1,
            "transmission": 0.1,
            "owner": 0.1,
        }

    # Tính khoảng cách Euclidean cho thuộc tính số đã chuẩn hóa
    sim_num = 0
    for col in ["selling_price", "km_driven", "year"]:
        diff = np.abs(
            case_df[f"{col}_scaled"] - query_item[f"{col}_scaled"]
        )
        sim_num += weights[col] * (1 - diff)

    # Tính khoảng cách Match/Exact (Hamming) cho thuộc tính phân loại
    sim_cat = 0
    for col in ["fuel", "transmission", "owner"]:
        sim_cat += weights[col] * (case_df[col] == query_item[col]).astype(int)

    total_sim = sim_num + sim_cat
    return total_sim


# 6. Bộ xử lý Critiquing (Critique Engine)
def apply_critique(case_df, current_item, critique_type):
    """Lọc dữ liệu dựa trên câu lệnh phê bình của người dùng"""
    filtered_df = case_df.copy()
    if critique_type == "cheaper":  # Rẻ hơn
        filtered_df = filtered_df[
            filtered_df["selling_price"] < current_item["selling_price"]
        ]
    elif critique_type == "less_km":  # Đi ít km hơn
        filtered_df = filtered_df[
            filtered_df["km_driven"] < current_item["km_driven"]
        ]
    elif critique_type == "newer":  # Năm sản xuất mới hơn
        filtered_df = filtered_df[filtered_df["year"] > current_item["year"]]
    return filtered_df

In [ ]:
# 7. Giả lập kịch bản người dùng tương tác để so sánh Baseline và Mô hình đề xuất
np.random.seed(42)
n_simulations = 100  # Mô phỏng 100 phiên tìm kiếm
top_k = 5

baseline_precisions = []
proposed_precisions = []
proposed_session_lengths = []

for _ in range(n_simulations):
    # Chọn ngẫu nhiên 1 mục tiêu mong muốn (Target Item)
    target_idx = np.random.choice(len(case_base))
    target_item = case_base.iloc[target_idx]

    # Item bắt đầu ngẫu nhiên
    start_idx = np.random.choice(len(case_base))
    current_item = case_base.iloc[start_idx]

    # BASELINE: Chỉ lấy KNN/Euclidean 1 lần, không có Critiquing 
    sim_scores_base = calculate_similarity(current_item, case_base)
    top_base_indices = sim_scores_base.nlargest(top_k + 1).index[1:]
    # Tính Precision@5 
    hits_base = sum(
        abs(case_base.loc[idx, "selling_price"] - target_item["selling_price"])
        / target_item["selling_price"]
        < 0.15
        for idx in top_base_indices
    )
    baseline_precisions.append(hits_base / top_k)

    # MÔ HÌNH ĐỀ XUẤT: CBR + Critiquing trong nhiều bước (Session)
    curr_df = case_base.copy()
    steps = 0
    max_steps = 10

    while steps < max_steps:
        steps += 1
        # Người dùng phê bình dựa trên tiêu chí còn lệch nhiều nhất so với target
        if current_item["selling_price"] > target_item["selling_price"]:
            curr_df = apply_critique(curr_df, current_item, "cheaper")
        elif current_item["km_driven"] > target_item["km_driven"]:
            curr_df = apply_critique(curr_df, current_item, "less_km")
        elif current_item["year"] < target_item["year"]:
            curr_df = apply_critique(curr_df, current_item, "newer")
        else:
            break  # Đã thỏa mãn tiêu chí

        if len(curr_df) == 0:  # Rủi ro tập rỗng
            break

        # Re-rank lại tập kết quả sau phê bình
        sim_scores = calculate_similarity(current_item, curr_df)
        best_idx = sim_scores.idxmax()
        current_item = curr_df.loc[best_idx]

    proposed_session_lengths.append(steps)

    # Tính Precision@5 
    final_sim = calculate_similarity(current_item, case_base)
    top_prop_indices = final_sim.nlargest(top_k).index
    hits_prop = sum(
        abs(case_base.loc[idx, "selling_price"] - target_item["selling_price"])
        / target_item["selling_price"]
        < 0.15
        for idx in top_prop_indices
    )
    proposed_precisions.append(hits_prop / top_k)

print("KẾT QUẢ THỰC NGHIỆM SƠ BỘ")
print(f"Precision@{top_k} Baseline: {np.mean(baseline_precisions):.2%}")
print(f"Precision@{top_k} Mô hình đề xuất: {np.mean(proposed_precisions):.2%}")
print(
    f"Số bước phê bình trung bình (Session Length): {np.mean(proposed_session_lengths):.1f} bước"
)

KẾT QUẢ THỰC NGHIỆM SƠ BỘ
Precision@5 Baseline: 10.80%
Precision@5 Mô hình đề xuất: 25.40%
Số bước phê bình trung bình (Session Length): 7.9 bước


In [5]:
# 8. Trích xuất bảng kết quả 
eval_summary = pd.DataFrame(
    {
        "Độ đo đánh giá": [
            f"Precision@{top_k}",
            "Session Length (Số bước trung bình)",
        ],
        "Baseline (KNN / Pure Euclidean)": [
            f"{np.mean(baseline_precisions):.1%}",
            "N/A (1 bước tĩnh)",
        ],
        "Mô hình đề xuất (CBR + Critiquing)": [
            f"{np.mean(proposed_precisions):.1%}",
            f"{np.mean(proposed_session_lengths):.1f} bước",
        ],
    }
)

eval_summary

,Độ đo đánh giá,Baseline (KNN / Pure Euclidean),Mô hình đề xuất (CBR + Critiquing)
0,Precision@5,10.8%,25.4%
1,Session Length (Số bước trung bình),N/A (1 bước tĩnh),7.9 bước
